# Detección de Fraude en Solicitudes de Crédito

## Objetivos de Aprendizaje

En este notebook, aprenderás a:

1. **Detectar solicitudes fraudulentas** con `SNOWFLAKE.ML.CLASSIFICATION`
2. **Identificar patrones de fraude documental** (identidades falsas, ingresos inflados)
3. **Cruzar datos de solicitud** con listas negras y patrones conocidos
4. **Calcular scores de riesgo de fraude** en la fase de originación
5. **Construir un dashboard** de alertas de fraude en solicitudes

---

## Lo Que Construirás

Un **sistema de detección de fraude en originación** que identifica solicitudes sospechosas:
- Modelo de clasificación binaria (fraude/legítimo) sobre solicitudes de crédito
- Features de inconsistencia: discrepancias entre datos declarados y verificados
- Detección de identidades sintéticas y datos fabricados
- Dashboard de alertas con priorización

**Valor de Negocio**: Prevenir pérdidas por aprobación de créditos fraudulentos (first-party fraud).

---

## Requisitos Técnicos

- **Cuenta Snowflake** con Cortex ML habilitado
- **Aproximadamente 15 minutos** para completar
- **100% SQL** (excepto dashboard Streamlit)

---

## Funcionalidades Clave

- `SNOWFLAKE.ML.CLASSIFICATION` — Modelo de detección de fraude
- `GENERATOR()` — Datos sintéticos de solicitudes
- `ABS()`, `DATEDIFF()` — Cálculo de discrepancias
- `OBJECT_CONSTRUCT` — Predicción inline

¡Comencemos!

---

## Paso 1: Configuración del Entorno

### Fraude en Originación

**Objetivo**: Detectar solicitudes de crédito fraudulentas antes de la aprobación.

### Tipos de Fraude en Solicitudes

- **Identidad sintética**: Datos inventados que no corresponden a persona real
- **Ingresos inflados**: Declaración de ingresos superiores a los reales
- **Empleo ficticio**: Empresas inexistentes o contactos falsos
- **Dirección falsa**: Domicilios que no coinciden con registros

In [ ]:
CREATE DATABASE IF NOT EXISTS FRAUDE_SOLICITUDES_DB;
CREATE SCHEMA IF NOT EXISTS FRAUDE_SOLICITUDES_DB.PUBLIC;
USE SCHEMA FRAUDE_SOLICITUDES_DB.PUBLIC;

CREATE WAREHOUSE IF NOT EXISTS COMPUTE_WH 
    WITH WAREHOUSE_SIZE = 'SMALL'
    AUTO_SUSPEND = 60 AUTO_RESUME = TRUE;

USE WAREHOUSE COMPUTE_WH;

SELECT CURRENT_DATABASE() AS db, CURRENT_SCHEMA() AS schema;

---

## Paso 2: Definir Estructura de Datos

### Tablas

1. **`SOLICITUDES_CREDITO`** — Datos declarados por el solicitante
2. **`VERIFICACIONES`** — Datos verificados por fuentes externas
3. **`LISTAS_NEGRAS`** — Registros de fraudes anteriores

In [ ]:
CREATE OR REPLACE TABLE SOLICITUDES_CREDITO (
    solicitud_id VARCHAR(10) PRIMARY KEY,
    fecha_solicitud DATE,
    nombre VARCHAR(100),
    dni VARCHAR(12),
    edad INTEGER,
    ingresos_declarados DECIMAL(10,2),
    empresa_declarada VARCHAR(100),
    antiguedad_empleo INTEGER,
    importe_solicitado DECIMAL(12,2),
    finalidad VARCHAR(50),
    email VARCHAR(100),
    telefono VARCHAR(15),
    codigo_postal VARCHAR(5),
    es_fraude BOOLEAN
);

CREATE OR REPLACE TABLE VERIFICACIONES (
    solicitud_id VARCHAR(10) PRIMARY KEY,
    ingresos_verificados DECIMAL(10,2),
    empresa_verificada BOOLEAN,
    direccion_verificada BOOLEAN,
    dni_valido BOOLEAN,
    score_buro INTEGER,
    antiguedad_direccion INTEGER,
    FOREIGN KEY (solicitud_id) REFERENCES SOLICITUDES_CREDITO(solicitud_id)
);

CREATE OR REPLACE TABLE LISTAS_NEGRAS (
    dni VARCHAR(12) PRIMARY KEY,
    motivo VARCHAR(100),
    fecha_registro DATE
);

SELECT 'Tablas creadas' AS status;

---

## Paso 3: Generar Datos Sintéticos

### Qué Vamos a Crear

- **8.000 solicitudes** (7% fraudulentas)
- **Verificaciones** con discrepancias para solicitudes fraudulentas
- **500 registros** en listas negras

### Patrones de Fraude

- Discrepancia >40% entre ingresos declarados y verificados
- Empresa no verificable
- DNI con formato inválido
- Score de buró bajo (<400)

In [ ]:
-- Generar solicitudes
INSERT INTO SOLICITUDES_CREDITO
SELECT
    'SOL' || LPAD(SEQ4()::STRING, 6, '0'),
    DATEADD('day', -UNIFORM(0, 365, RANDOM()), CURRENT_DATE()),
    'SOLICITANTE_' || SEQ4(),
    LPAD(UNIFORM(10000000, 99999999, RANDOM())::STRING, 8, '0') || ARRAY_CONSTRUCT('A','B','C','D','E','F','G','H','J','K')[UNIFORM(0,9,RANDOM())]::VARCHAR,
    UNIFORM(21, 70, RANDOM()),
    ROUND(UNIFORM(1200, 8000, RANDOM()), 2),
    'EMPRESA_' || UNIFORM(1, 500, RANDOM()),
    UNIFORM(0, 25, RANDOM()),
    ROUND(UNIFORM(3000, 80000, RANDOM()), 2),
    ARRAY_CONSTRUCT('Consumo','Vehículo','Vivienda','Negocio','Estudios','Consolidación')[UNIFORM(0,5,RANDOM())]::VARCHAR,
    'email_' || SEQ4() || '@test.com',
    '6' || LPAD(UNIFORM(10000000, 99999999, RANDOM())::STRING, 8, '0'),
    LPAD(UNIFORM(1000, 52999, RANDOM())::STRING, 5, '0'),
    CASE WHEN UNIFORM(0,100,RANDOM()) < 7 THEN TRUE ELSE FALSE END
FROM TABLE(GENERATOR(ROWCOUNT => 8000));

-- Generar verificaciones (discrepancias para fraude)
INSERT INTO VERIFICACIONES
SELECT
    s.solicitud_id,
    CASE WHEN s.es_fraude THEN ROUND(s.ingresos_declarados * UNIFORM(30, 60, RANDOM()) / 100, 2)
         ELSE ROUND(s.ingresos_declarados * UNIFORM(85, 115, RANDOM()) / 100, 2) END,
    CASE WHEN s.es_fraude THEN UNIFORM(0,1,RANDOM()) < 0.3 ELSE TRUE END,
    CASE WHEN s.es_fraude THEN UNIFORM(0,1,RANDOM()) < 0.4 ELSE TRUE END,
    CASE WHEN s.es_fraude THEN UNIFORM(0,1,RANDOM()) < 0.5 ELSE TRUE END,
    CASE WHEN s.es_fraude THEN UNIFORM(200, 450, RANDOM()) ELSE UNIFORM(550, 850, RANDOM()) END,
    CASE WHEN s.es_fraude THEN UNIFORM(0, 3, RANDOM()) ELSE UNIFORM(2, 20, RANDOM()) END
FROM SOLICITUDES_CREDITO s;

SELECT es_fraude, COUNT(*), ROUND(AVG(ingresos_declarados),0) AS ing_medio
FROM SOLICITUDES_CREDITO GROUP BY 1;

---

## Paso 4: Ingeniería de Variables de Fraude

### Features de Discrepancia

| Variable | Descripción | Señal de Fraude |
|---|---|---|
| `discrepancia_ingresos` | % diferencia declarado vs verificado | >40% |
| `empresa_no_verificada` | Empresa no encontrada en registros | TRUE |
| `score_buro_bajo` | Score crediticio <400 | TRUE |
| `direccion_reciente` | <1 año en dirección actual | TRUE |

In [ ]:
-- Feature engineering de fraude en solicitudes
CREATE OR REPLACE TABLE FEATURES_FRAUDE_SOL AS
SELECT
    s.solicitud_id,
    s.edad::FLOAT,
    s.ingresos_declarados::FLOAT,
    s.importe_solicitado::FLOAT,
    s.antiguedad_empleo::FLOAT,
    (s.importe_solicitado / NULLIF(s.ingresos_declarados, 0))::FLOAT AS ratio_importe_ingreso,
    ABS(s.ingresos_declarados - v.ingresos_verificados)::FLOAT / NULLIF(s.ingresos_declarados, 0) AS discrepancia_ingresos,
    CASE WHEN v.empresa_verificada THEN 0 ELSE 1 END::FLOAT AS empresa_no_verificada,
    CASE WHEN v.direccion_verificada THEN 0 ELSE 1 END::FLOAT AS direccion_no_verificada,
    CASE WHEN v.dni_valido THEN 0 ELSE 1 END::FLOAT AS dni_invalido,
    v.score_buro::FLOAT,
    v.antiguedad_direccion::FLOAT,
    (empresa_no_verificada + direccion_no_verificada + dni_invalido)::FLOAT AS total_discrepancias,
    s.es_fraude
FROM SOLICITUDES_CREDITO s
JOIN VERIFICACIONES v ON s.solicitud_id = v.solicitud_id;

SELECT * FROM FEATURES_FRAUDE_SOL LIMIT 10;

---

## Paso 5: Entrenar y Evaluar Modelo

### Modelo de Detección de Fraude en Originación

El clasificador aprenderá a identificar solicitudes fraudulentas basándose en discrepancias entre datos declarados y verificados.

In [ ]:
-- Dividir y entrenar
CREATE OR REPLACE TABLE TRAIN_FRAUDE_SOL AS SELECT * FROM FEATURES_FRAUDE_SOL SAMPLE (80);
CREATE OR REPLACE TABLE TEST_FRAUDE_SOL AS SELECT * FROM FEATURES_FRAUDE_SOL WHERE solicitud_id NOT IN (SELECT solicitud_id FROM TRAIN_FRAUDE_SOL);

CREATE OR REPLACE SNOWFLAKE.ML.CLASSIFICATION detector_fraude_sol(
    INPUT_DATA => SYSTEM$REFERENCE('TABLE', 'TRAIN_FRAUDE_SOL'),
    TARGET_COLNAME => 'ES_FRAUDE'
);

In [ ]:
CALL detector_fraude_sol!SHOW_EVALUATION_METRICS();
CALL detector_fraude_sol!SHOW_FEATURE_IMPORTANCE();

---

## Paso 6: Puntuar Solicitudes

### Niveles de Alerta

- **BLOQUEAR** (≥75%): Solicitud rechazada automáticamente
- **INVESTIGAR** (≥40%): Revisión manual del equipo antifraude
- **APROBAR** (<40%): Continuar proceso normal

In [ ]:
-- Puntuar solicitudes
CREATE OR REPLACE TABLE ALERTAS_FRAUDE_SOL AS
SELECT
    solicitud_id, ingresos_declarados, importe_solicitado,
    discrepancia_ingresos, total_discrepancias, score_buro,
    es_fraude AS fraude_real,
    detector_fraude_sol!PREDICT(
        OBJECT_CONSTRUCT(
            'EDAD', edad, 'INGRESOS_DECLARADOS', ingresos_declarados,
            'IMPORTE_SOLICITADO', importe_solicitado,
            'ANTIGUEDAD_EMPLEO', antiguedad_empleo,
            'RATIO_IMPORTE_INGRESO', ratio_importe_ingreso,
            'DISCREPANCIA_INGRESOS', discrepancia_ingresos,
            'EMPRESA_NO_VERIFICADA', empresa_no_verificada,
            'DIRECCION_NO_VERIFICADA', direccion_no_verificada,
            'DNI_INVALIDO', dni_invalido,
            'SCORE_BURO', score_buro,
            'ANTIGUEDAD_DIRECCION', antiguedad_direccion,
            'TOTAL_DISCREPANCIAS', total_discrepancias
        )
    ) AS pred,
    ROUND(pred['probability']['true']::FLOAT * 100, 1) AS prob_fraude,
    CASE
        WHEN pred['probability']['true']::FLOAT >= 0.75 THEN 'BLOQUEAR'
        WHEN pred['probability']['true']::FLOAT >= 0.40 THEN 'INVESTIGAR'
        ELSE 'APROBAR'
    END AS decision
FROM TEST_FRAUDE_SOL;

SELECT decision, COUNT(*) AS n, ROUND(AVG(prob_fraude),1) AS prob_media
FROM ALERTAS_FRAUDE_SOL GROUP BY 1 ORDER BY 2 DESC;

---

## Paso 7: Dashboard de Alertas

### Panel Antifraude

Dashboard para el equipo antifraude con alertas y métricas de detección.

In [ ]:
import streamlit as st
import pandas as pd
from snowflake.snowpark.context import get_active_session

session = get_active_session()

st.title('Detección de Fraude en Solicitudes')
st.markdown('### Panel Antifraude — Originación')

total = session.sql('SELECT COUNT(*) FROM ALERTAS_FRAUDE_SOL').collect()[0][0]
bloq = session.sql("SELECT COUNT(*) FROM ALERTAS_FRAUDE_SOL WHERE decision='BLOQUEAR'").collect()[0][0]
invest = session.sql("SELECT COUNT(*) FROM ALERTAS_FRAUDE_SOL WHERE decision='INVESTIGAR'").collect()[0][0]

c1, c2, c3 = st.columns(3)
c1.metric('Total Solicitudes', f'{total:,}')
c2.metric('Bloqueadas', f'{bloq:,}')
c3.metric('A Investigar', f'{invest:,}')

st.markdown('---')
df = session.sql('SELECT decision, COUNT(*) AS n FROM ALERTAS_FRAUDE_SOL GROUP BY 1').to_pandas()
st.bar_chart(df.set_index('DECISION'))

st.markdown('---')
st.subheader('Solicitudes Sospechosas')
df_sosp = session.sql("SELECT solicitud_id, prob_fraude, discrepancia_ingresos, total_discrepancias, score_buro, decision FROM ALERTAS_FRAUDE_SOL WHERE decision != 'APROBAR' ORDER BY prob_fraude DESC LIMIT 30").to_pandas()
st.dataframe(df_sosp)

---

## Paso 8: Limpieza del Entorno (Opcional)

⚠️ **Advertencia**: Eliminará todos los datos y modelos.

In [ ]:
-- Descomentar para limpiar

-- DROP DATABASE IF EXISTS FRAUDE_SOLICITUDES_DB;